# CVRP Solver - Multi-Depot Support

This notebook contains a CVRP solver with:
- Multi-depot support with configurable reload strategies
- Route extraction with depot movements
- Load progression calculation
- Graph-based route trajectory computation
- Edge load aggregation for infrastructure analysis
- Fixed edge load calculation for correct load distribution


In [ ]:
import pyvrp
import pyvrp.plotting
import pyvrp.stop
import osmnx as ox
import random
import networkx as nx
import matplotlib.pyplot as plt
import igraph as ig
from typing import Tuple, Dict, List, Union
from shapely import wkt
from shapely.geometry import Polygon, Point
import geopandas as gpd
import pandas as pd
import logging
from collections import defaultdict


## Section 1: Graph Utilities

Functions for graph conversion and manipulation.


In [ ]:
def networkx_to_igraph_with_indices(g: nx.MultiDiGraph) -> Tuple[ig.Graph, Dict[str, dict]]:
    """Convert networkx graph to igraph with bidirectional index mappings.
    
    Args:
        g: NetworkX MultiDiGraph
    
    Returns:
        Tuple of (igraph Graph, index mappings dict)
        Index mappings contain: node_nx_to_ig, node_ig_to_nx, edge_nx_to_ig, edge_ig_to_nx
    """
    e = ox.graph_to_gdfs(g, nodes=False, edges=True)
    nx.set_edge_attributes(g, {idx: idx for idx in e.index}, name="nx_edge_id")
    h = ig.Graph.from_networkx(g)
    
    idx_maps = {
        "node_nx_to_ig": {a: b for a, b in zip(h.vs()["_nx_name"], h.vs.indices)},
        "node_ig_to_nx": {b: a for a, b in zip(h.vs()["_nx_name"], h.vs.indices)},
        "edge_nx_to_ig": {a: b for a, b in zip(h.es()["nx_edge_id"], h.get_edgelist())},
        "edge_ig_to_nx": {b: a for a, b in zip(h.es()["nx_edge_id"], h.get_edgelist())},
    }
    
    return h, idx_maps


In [ ]:
def nx_to_ig(graph: nx.MultiDiGraph) -> ig.Graph:
    """Simple NetworkX to igraph conversion.
    
    Args:
        graph: NetworkX MultiDiGraph
    
    Returns:
        igraph Graph
    """
    e = ox.graph_to_gdfs(graph, nodes=False, edges=True)
    nx.set_edge_attributes(graph, {idx: idx for idx in e.index}, name="nx_edge_id")
    g_ig = ig.Graph.from_networkx(graph)
    return g_ig


In [ ]:
def get_intersection_nodes(graph: nx.MultiDiGraph) -> List:
    """Get list of intersection nodes (degree > 2).
    
    Args:
        graph: NetworkX MultiDiGraph
    
    Returns:
        List of node IDs that are intersections
    """
    return [node for node, degree in graph.degree() if degree > 2]


## Section 2: Data Preparation

Functions for loading and preparing waste collection data.


In [ ]:
# Constants
LAUSANNE_HULL_WKT = (
    "POLYGON ((6.6973149 46.4991489, 6.5474464 46.508319, 6.5441708 46.5091808, "
    "6.5429015 46.5134131, 6.5361596 46.5588201, 6.5949948 46.5926757, "
    "6.6582251 46.5991119, 6.7121747 46.571259, 6.7180473 46.5236707, "
    "6.7159339 46.5179337, 6.7123702 46.5099251, 6.7077559 46.5021425, "
    "6.7016307 46.4999401, 6.6973149 46.4991489))"
)


In [ ]:
def process_centroids(
    graph: nx.MultiDiGraph, 
    idx_maps: dict, 
    waste_per_centroid: int = 10,
    centroid_csv: str = "data/DI_final_clustered_centroids.csv"
) -> pd.DataFrame:
    """Process waste centroids and map to nearest graph nodes.
    
    Args:
        graph: NetworkX MultiDiGraph
        idx_maps: Index mapping dictionary from networkx_to_igraph_with_indices
        waste_per_centroid: Weight of waste per centroid in kg
        centroid_csv: Path to centroid CSV file
    
    Returns:
        DataFrame with columns: node, centroid_waste, x, y, node_ig
    """
    df = pd.read_csv(centroid_csv)
    gdf = gpd.GeoDataFrame(
        df, 
        geometry=df.apply(lambda row: Point(row["centroid_lon"], row["centroid_lat"]), axis=1)
    )
    hull: Polygon = wkt.loads(LAUSANNE_HULL_WKT)
    
    filtered_gdf = gdf[gdf["geometry"].apply(lambda geom: geom.within(hull))].copy()
    filtered_gdf["node"] = ox.distance.nearest_nodes(
        graph, 
        filtered_gdf["centroid_lon"], 
        filtered_gdf["centroid_lat"]
    )
    filtered_gdf["centroid_waste"] = waste_per_centroid
    
    node_df = filtered_gdf.groupby("node")["centroid_waste"].sum().reset_index()
    
    # Add coordinates
    node_df["x"] = node_df["node"].map(lambda n: graph.nodes[n]["x"])
    node_df["y"] = node_df["node"].map(lambda n: graph.nodes[n]["y"])
    
    # Convert to igraph indices
    node_df["node_ig"] = node_df["node"].map(lambda n: idx_maps["node_nx_to_ig"][n])
    
    return node_df


In [ ]:
def add_depots(
    node_df: pd.DataFrame, 
    graph: nx.MultiDiGraph, 
    idx_maps: dict,
    depot_locations: Union[Tuple[float, float], List[Tuple[float, float]]] = None
) -> Tuple[pd.DataFrame, List[int]]:
    """Add one or more depot locations to node DataFrame.
    
    Args:
        node_df: DataFrame with client nodes
        graph: NetworkX MultiDiGraph
        idx_maps: Index mapping dictionary
        depot_locations: Single (lon, lat) tuple or list of tuples for multiple depots
                        If None, uses default single depot at (6.6328, 46.5225)
    
    Returns:
        Tuple of (updated DataFrame with depot(s), list of depot igraph node indices)
    """
    # Handle default and normalize to list
    if depot_locations is None:
        depot_locations = [(6.6328, 46.5225)]
    elif isinstance(depot_locations, tuple):
        depot_locations = [depot_locations]
    
    node_df["isdepot"] = False
    node_df["depot_id"] = None
    
    depot_node_igs = []
    
    for depot_id, (center_lon, center_lat) in enumerate(depot_locations):
        depot_node_nx = ox.distance.nearest_nodes(graph, center_lon, center_lat)
        depot_node_ig = idx_maps["node_nx_to_ig"][depot_node_nx]
        depot_node_igs.append(depot_node_ig)
        
        # If depot node already exists in node_df, mark it
        if depot_node_nx in node_df["node"].values:
            node_df.loc[node_df["node"] == depot_node_nx, "centroid_waste"] = 0
            node_df.loc[node_df["node"] == depot_node_nx, "isdepot"] = True
            node_df.loc[node_df["node"] == depot_node_nx, "depot_id"] = depot_id
        else:
            # Add new row for depot
            depot_df = pd.DataFrame({
                "node": [depot_node_nx],
                "node_ig": [depot_node_ig],
                "centroid_waste": [0],
                "x": [graph.nodes()[depot_node_nx]["x"]],
                "y": [graph.nodes()[depot_node_nx]["y"]],
                "isdepot": [True],
                "depot_id": [depot_id]
            })
            node_df = pd.concat([depot_df, node_df], ignore_index=True)
    
    return node_df, depot_node_igs


## Section 3: CVRP Model Creation & Solving

Functions for creating and solving the CVRP problem with multi-depot support.


In [ ]:
def create_distance_matrix(
    g_ig: ig.Graph, 
    node_df: pd.DataFrame
) -> Tuple[List, List]:
    """Create origin-destination distance matrix.
    
    Args:
        g_ig: igraph Graph
        node_df: DataFrame with node_ig column (including depot(s))
    
    Returns:
        Tuple of (distance matrix, list of inaccessible node indices)
    """
    unique_ig_nodes = list(set(node_df["node_ig"].tolist()))
    od_distance = g_ig.distances(unique_ig_nodes, unique_ig_nodes, weights="length")
    
    # Check for inaccessible nodes from first depot
    inaccessible_indices = [
        i for i, dist in enumerate(od_distance[0]) 
        if dist == float('inf')
    ]
    
    return od_distance, inaccessible_indices


In [ ]:
def create_cvrp_model(
    node_df: pd.DataFrame,
    inaccessible_indices: List,
    od_distance: List,
    n_vehicles: Union[int, List[int]] = 15,
    vehicle_capacity: int = 5000,
    reload_strategy: str = "all_depots"
) -> pyvrp.Model:
    """Create PyVRP model with single or multi-depot support.
    
    Args:
        node_df: DataFrame with depot(s) and client nodes
        inaccessible_indices: List of nodes unreachable from first depot
        od_distance: Origin-destination distance matrix
        n_vehicles: Number of vehicles (int for single depot, list for multi-depot)
                   E.g., [5, 3, 2] assigns 5 vehicles to depot 0, 3 to depot 1, 2 to depot 2
        vehicle_capacity: Maximum capacity per vehicle in kg
        reload_strategy: "all_depots" = vehicles can reload at any depot
                        "home_only" = vehicles only reload at their home depot
    
    Returns:
        Configured PyVRP Model
    """
    m = pyvrp.Model()
    
    # Get depot rows sorted by depot_id
    depot_rows = node_df[node_df["isdepot"]].sort_values("depot_id")
    n_depots = len(depot_rows)
    
    # Normalize n_vehicles to list
    if isinstance(n_vehicles, int):
        n_vehicles_list = [n_vehicles] if n_depots == 1 else [n_vehicles // n_depots] * n_depots
    else:
        n_vehicles_list = n_vehicles
    
    if len(n_vehicles_list) != n_depots:
        raise ValueError(f'n_vehicles list length ({len(n_vehicles_list)}) must match number of depots ({n_depots})')
    
    # Add all depots
    depots = []
    for _, depot_row in depot_rows.iterrows():
        depot = m.add_depot(
            x=depot_row["x"],
            y=depot_row["y"]
        )
        depots.append(depot)
    
    # Create vehicle type for each depot
    regular = m.add_profile(name="regular")
    
    for depot_idx, (depot, n_veh) in enumerate(zip(depots, n_vehicles_list)):
        if n_veh == 0:
            continue
        
        # Determine reload depots based on strategy
        if reload_strategy == "home_only":
            reload_depots = [depot]
        elif reload_strategy == "all_depots":
            reload_depots = depots
        else:
            raise ValueError(f'Invalid reload_strategy: {reload_strategy}. Use "home_only" or "all_depots"')
        
        m.add_vehicle_type(
            n_veh,
            capacity=vehicle_capacity,
            start_depot=depot,
            end_depot=depot,
            reload_depots=reload_depots,
            max_reloads=10,
            profile=regular
        )
    
    # Add clients
    for _, row in node_df.iterrows():
        if not row["isdepot"]:
            required = row["node_ig"] not in inaccessible_indices
            
            m.add_client(
                x=row["x"], 
                y=row["y"], 
                delivery=int(row["centroid_waste"]),
                required=required
            )
    
    # Add edges
    for frm_idx, frm in enumerate(m.locations):
        for to_idx, to in enumerate(m.locations):
            duration = od_distance[frm_idx][to_idx]
            
            if duration == float('inf'):
                duration = 999999
            
            m.add_edge(frm, to, distance=duration)
    
    return m


In [ ]:
def solve_cvrp(
    model: pyvrp.Model, 
    max_runtime: int = 5, 
    no_improvement_iterations: int = 100, 
    seed: int = 42
) -> pyvrp.Result:
    """Solve CVRP model.
    
    Args:
        model: PyVRP Model to solve
        max_runtime: Maximum runtime in seconds
        no_improvement_iterations: Stop after N iterations with no improvement
        seed: Random seed for reproducibility
    
    Returns:
        PyVRP Result object
    """
    stop_criteria = pyvrp.stop.MultipleCriteria([
        pyvrp.stop.MaxRuntime(max_runtime),
        pyvrp.stop.NoImprovement(no_improvement_iterations)
    ])
    
    result = model.solve(stop=stop_criteria, seed=seed)
    return result


## Section 4: Route Processing & Analysis

Functions for extracting routes, calculating loads, and mapping to graph.


In [ ]:
def extract_routes_with_depots(result: pyvrp.Result) -> List[Dict]:
    """Extract routes with explicit depot visits.
    
    Args:
        result: PyVRP Result object
    
    Returns:
        List of route dictionaries:
        [
            {
                'route_id': 0,
                'vehicle_id': 0,
                'start_depot_id': 0,
                'end_depot_id': 0,
                'total_distance': 12345.6,
                'total_delivery': 500,
                'n_trips': 2,
                'trips': [
                    {
                        'trip_id': 0,
                        'sequence': [0, 25, 423, 12, 0],
                        'distance': 5000.0,
                        'delivery': 250,
                        'start_depot': 0,
                        'end_depot': 0,
                        'is_reload': False
                    },
                    ...
                ]
            },
            ...
        ]
    """
    result_routes = []
    
    for route_id, route in enumerate(result.best.routes()):
        total_distance = 0
        total_delivery = 0
        trips_data = []
        
        for trip_id, trip in enumerate(route.trips()):
            # Build sequence with depot at start and end
            sequence = [trip.start_depot()] + list(trip.visits()) + [trip.end_depot()]
            
            trip_distance = trip.distance()
            trip_delivery = trip.delivery()[0]  # First dimension is delivery
            
            total_distance += trip_distance
            total_delivery += trip_delivery
            
            trips_data.append({
                'trip_id': trip_id,
                'sequence': sequence,
                'distance': trip_distance,
                'delivery': trip_delivery,
                'start_depot': trip.start_depot(),
                'end_depot': trip.end_depot(),
                'is_reload': trip_id > 0  # Trips after first are reloads
            })
        
        # Determine depot IDs from first and last trip
        start_depot_id = trips_data[0]['start_depot'] if trips_data else 0
        end_depot_id = trips_data[-1]['end_depot'] if trips_data else 0
        
        result_routes.append({
            'route_id': route_id,
            'vehicle_id': route_id,
            'start_depot_id': start_depot_id,
            'end_depot_id': end_depot_id,
            'total_distance': total_distance,
            'total_delivery': total_delivery,
            'n_trips': len(trips_data),
            'trips': trips_data
        })
    
    return result_routes


In [ ]:
def calculate_load_progression(
    routes: List[Dict], 
    node_df: pd.DataFrame
) -> List[Dict]:
    """Calculate cumulative load after pickup at each client location.
    
    Args:
        routes: List of route dictionaries from extract_routes_with_depots()
        node_df: DataFrame with client demands (centroid_waste column)
    
    Returns:
        List of load progression dictionaries:
        [
            {
                'route_id': 0,
                'trip_id': 0,
                'progression': [
                    {'location_idx': 0, 'cumulative_load': 0},  # depot start
                    {'location_idx': 25, 'cumulative_load': 10},
                    {'location_idx': 423, 'cumulative_load': 20},
                    ...
                    {'location_idx': 0, 'cumulative_load': 0}  # depot end (after unload)
                ]
            },
            ...
        ]
    """
    # Create lookup for client demands by location index
    # Location indices 0, 1, 2, ... are depots, rest are clients
    n_depots = node_df["isdepot"].sum()
    
    demands = {}
    # All depots have 0 demand
    for i in range(n_depots):
        demands[i] = 0
    
    # Clients start at index n_depots
    client_idx = n_depots
    for _, row in node_df.iterrows():
        if not row.get("isdepot", False):
            demands[client_idx] = row["centroid_waste"]
            client_idx += 1
    
    result = []
    
    for route in routes:
        route_id = route['route_id']
        
        for trip in route['trips']:
            trip_id = trip['trip_id']
            sequence = trip['sequence']
            
            progression = []
            cumulative_load = 0
            
            for loc_idx in sequence:
                # At any depot end, unload everything
                if loc_idx < n_depots and len(progression) > 0:
                    cumulative_load = 0
                # At clients, add pickup
                elif loc_idx >= n_depots:
                    cumulative_load += demands.get(loc_idx, 0)
                
                progression.append({
                    'location_idx': loc_idx,
                    'cumulative_load': cumulative_load
                })
            
            result.append({
                'route_id': route_id,
                'trip_id': trip_id,
                'progression': progression
            })
    
    return result


In [ ]:
def route_on_graph(
    routes: List[Dict], 
    g_ig: ig.Graph, 
    node_df: pd.DataFrame,
    idx_maps: Dict
) -> Dict:
    """Route each trip segment on the actual street network.
    
    Args:
        routes: List of route dictionaries from extract_routes_with_depots()
        g_ig: igraph Graph
        node_df: DataFrame with node mappings
        idx_maps: Index mapping dictionary
    
    Returns:
        Dictionary with:
        {
            'routes': routes,  # Pass-through of input routes
            'graph_paths': [
                {
                    'route_id': 0,
                    'trip_id': 0,
                    'segment_id': 0,
                    'from_loc': 0,
                    'to_loc': 25,
                    'path_ig': [1234, 1235, ...],  # igraph node indices
                    'path_nx': [node1, node2, ...],  # NetworkX node IDs
                    'path_length': 1234.5,
                    'success': True
                },
                ...
            ],
            'routing_stats': {
                'total_segments': 1000,
                'successful': 993,
                'failed': 7,
                'failed_segments': [(route_id, trip_id, from_loc, to_loc), ...]
            }
        }
    """
    # Map VRP location indices to igraph node indices
    # Depots come first (0, 1, 2, ...), then clients
    depot_rows = node_df[node_df["isdepot"]].sort_values("depot_id")
    loc_to_ig = depot_rows["node_ig"].tolist()
    
    # Add clients
    for _, row in node_df.iterrows():
        if not row.get("isdepot", False):
            loc_to_ig.append(row["node_ig"])
    
    # Map igraph indices back to NetworkX node IDs
    ig_to_nx = idx_maps["node_ig_to_nx"]
    
    graph_paths = []
    total_segments = 0
    successful = 0
    failed = 0
    failed_segments = []
    
    for route in routes:
        route_id = route['route_id']
        
        for trip in route['trips']:
            trip_id = trip['trip_id']
            sequence = trip['sequence']
            
            for segment_id in range(len(sequence) - 1):
                from_loc = sequence[segment_id]
                to_loc = sequence[segment_id + 1]
                
                total_segments += 1
                
                try:
                    from_ig = loc_to_ig[from_loc]
                    to_ig = loc_to_ig[to_loc]
                    
                    # Get shortest path on graph
                    path_ig = g_ig.get_shortest_paths(
                        from_ig, 
                        to=to_ig, 
                        weights="length", 
                        output="vpath"
                    )[0]
                    
                    if len(path_ig) == 0:
                        # Empty path = unreachable
                        logging.warning(
                            f'Route {route_id}, Trip {trip_id}, Segment {segment_id}: '
                            f'No path found from location {from_loc} to {to_loc}'
                        )
                        failed += 1
                        failed_segments.append((route_id, trip_id, from_loc, to_loc))
                        
                        graph_paths.append({
                            'route_id': route_id,
                            'trip_id': trip_id,
                            'segment_id': segment_id,
                            'from_loc': from_loc,
                            'to_loc': to_loc,
                            'path_ig': [],
                            'path_nx': [],
                            'path_length': 0,
                            'success': False
                        })
                        continue
                    
                    # Convert to NetworkX node IDs
                    path_nx = [ig_to_nx[ig_idx] for ig_idx in path_ig]
                    
                    # Calculate path length
                    path_length = sum(
                        g_ig.es[g_ig.get_eid(path_ig[i], path_ig[i+1])]['length']
                        for i in range(len(path_ig) - 1)
                    )
                    
                    successful += 1
                    graph_paths.append({
                        'route_id': route_id,
                        'trip_id': trip_id,
                        'segment_id': segment_id,
                        'from_loc': from_loc,
                        'to_loc': to_loc,
                        'path_ig': path_ig,
                        'path_nx': path_nx,
                        'path_length': path_length,
                        'success': True
                    })
                    
                except Exception as e:
                    logging.error(f'Error routing segment: {e}')
                    failed += 1
                    failed_segments.append((route_id, trip_id, from_loc, to_loc))
                    
                    graph_paths.append({
                        'route_id': route_id,
                        'trip_id': trip_id,
                        'segment_id': segment_id,
                        'from_loc': from_loc,
                        'to_loc': to_loc,
                        'path_ig': [],
                        'path_nx': [],
                        'path_length': 0,
                        'success': False
                    })
    
    return {
        'routes': routes,
        'graph_paths': graph_paths,
        'routing_stats': {
            'total_segments': total_segments,
            'successful': successful,
            'failed': failed,
            'failed_segments': failed_segments
        }
    }


In [ ]:
def calculate_edge_loads(
    routing_result: Dict,
    load_progression: List[Dict],
    graph: nx.MultiDiGraph,
    unit: str = 'kg'
) -> Dict:
    """Calculate total load passing through each edge.
    
    Args:
        routing_result: Output from route_on_graph()
        load_progression: Output from calculate_load_progression()
        graph: NetworkX MultiDiGraph
        unit: 'kg' for load sum, or 'kg_m' for load × distance
    
    Returns:
        Dictionary with:
        {
            'edge_loads': {
                (node1, node2, key): total_load,
                ...
            },
            'max_load': 5000,
            'max_load_edge': (node1, node2, 0),
            'unit': 'kg',
            'n_edges_used': 1234
        }
    """
    edge_loads = defaultdict(float)
    
    # Create lookup for load at each segment
    segment_loads = {}
    for lp in load_progression:
        route_id = lp['route_id']
        trip_id = lp['trip_id']
        
        for i in range(len(lp['progression']) - 1):
            from_loc = lp['progression'][i]['location_idx']
            to_loc = lp['progression'][i+1]['location_idx']
            # FIXED: Load during this segment is the load at from_loc (while traveling)
            load = lp['progression'][i]['cumulative_load']
            
            segment_loads[(route_id, trip_id, i)] = load
    
    # Process each graph path
    for gp in routing_result['graph_paths']:
        if not gp['success']:
            continue
        
        route_id = gp['route_id']
        trip_id = gp['trip_id']
        segment_id = gp['segment_id']
        path_nx = gp['path_nx']
        
        # Get load for this segment
        load = segment_loads.get((route_id, trip_id, segment_id), 0)
        
        # Iterate through each edge in the path
        for i in range(len(path_nx) - 1):
            u = path_nx[i]
            v = path_nx[i+1]
            
            # Find the edge (there may be multiple edges between u and v in a MultiDiGraph)
            if graph.has_edge(u, v):
                # Get all edge keys between u and v
                edge_data = graph.get_edge_data(u, v)
                
                # Use the shortest edge (by length) for routing
                if len(edge_data) == 1:
                    key = list(edge_data.keys())[0]
                else:
                    # Multiple edges exist, choose the shortest one
                    key = min(edge_data.keys(), 
                            key=lambda k: edge_data[k].get('length', float('inf')))
                
                if unit == 'kg':
                    edge_loads[(u, v, key)] += load
                elif unit == 'kg_m':
                    length = edge_data[key].get('length', 0)
                    edge_loads[(u, v, key)] += load * length
            else:
                # Edge might be in reverse direction (common in MultiDiGraphs)
                if graph.has_edge(v, u):
                    edge_data = graph.get_edge_data(v, u)
                    if len(edge_data) == 1:
                        key = list(edge_data.keys())[0]
                    else:
                        key = min(edge_data.keys(), 
                                key=lambda k: edge_data[k].get('length', float('inf')))
                    
                    if unit == 'kg':
                        edge_loads[(v, u, key)] += load
                    elif unit == 'kg_m':
                        length = edge_data[key].get('length', 0)
                        edge_loads[(v, u, key)] += load * length
    
    # Find max load edge
    if edge_loads:
        max_load_edge = max(edge_loads.items(), key=lambda x: x[1])
        max_edge = max_load_edge[0]
        max_load = max_load_edge[1]
    else:
        max_edge = None
        max_load = 0
    
    return {
        'edge_loads': dict(edge_loads),
        'max_load': max_load,
        'max_load_edge': max_edge,
        'unit': unit,
        'n_edges_used': len(edge_loads)
    }


## Section 5: Complete Workflow Demonstration

Demonstration of the complete workflow from data loading to route analysis.


In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


In [ ]:
# Load graph
print("Loading street network graph...")
graph = ox.load_graphml("data/lausanne.graphml")
print(f"Graph loaded: {len(graph.nodes())} nodes, {len(graph.edges())} edges")


In [ ]:
# Convert graph
print("Converting graph to igraph...")
g_ig, idx_maps = networkx_to_igraph_with_indices(graph)
print("Graph conversion complete")


In [ ]:
# Process centroids
print("Processing waste centroids...")
waste_per_centroid = 10
node_df = process_centroids(graph, idx_maps, waste_per_centroid=waste_per_centroid)
print(f"Processed {len(node_df)} collection points")


### Configuration: Single or Multi-Depot

Uncomment one of the configurations below:


In [ ]:
# ========== SINGLE DEPOT CONFIGURATION (Default) ==========
depot_locations = (6.597982, 46.527867)  # Single tuple
n_vehicles = 5
reload_strategy = "all_depots"

# ========== MULTI-DEPOT CONFIGURATION (Example) ==========
# depot_locations = [
#     (6.597982, 46.527867),  # Depot 0: Recycling center
#     (6.6000, 46.5500),  # Depot 1: North district
#     (6.6500, 46.5000),  # Depot 2: South district
# ]
# n_vehicles = [3, 2, 2]  # 3 vehicles at depot 0, 2 at depot 1, 2 at depot 2
# reload_strategy = "home_only"  # or "all_depots"

# # lausanne_post_offices
depot_locations = [
    (6.634763001182209, 46.51786477124877, ), # train station
    (6.63484387895323, 46.52280567146395), # Riponne
    (6.629079677111272, 46.522894798355864), # Sévelin
    (6.616255947170737, 46.51567504637278), # Avenue de Cour
    (6.636592569361496, 46.51926275908995), # Saint-François
    (6.605310440911157, 46.52759603474043), # Malley
    (6.5894279622916585, 46.52474383288413), # Bourdonnette
    (6.6314276590463965, 46.53290073179973), # Pontaise
    (6.657349467139207, 46.52649108372711), # Chailly
    (6.650884859700522, 46.53385993374889), # Sallaz
    (6.640543342774893, 46.51189489849091), # Montchoisi
    (6.631788612292283, 46.50894136727424), # Ouchy
  ]

n_vehicles = [1 for _ in depot_locations]  # 1 vehicle at each depot
reload_strategy = "home_only"  # Vehicles can reload at any depot


In [ ]:
# Add depot(s)
print(f"Adding depot(s)...")
node_df, depot_node_igs = add_depots(node_df, graph, idx_maps, depot_locations)
n_depots = node_df["isdepot"].sum()
print(f"Added {n_depots} depot(s)")


In [ ]:
# Create distance matrix
print("Creating distance matrix...")
od_distance, inaccessible = create_distance_matrix(g_ig, node_df)
print(f"Distance matrix created: {len(od_distance)}x{len(od_distance)} matrix")
print(f"Inaccessible nodes: {len(inaccessible)}")


In [ ]:
# Create CVRP model
print("Creating CVRP model...")
vehicle_capacity = 5000
model = create_cvrp_model(
    node_df, 
    inaccessible, 
    od_distance, 
    n_vehicles=n_vehicles,
    vehicle_capacity=vehicle_capacity,
    reload_strategy=reload_strategy
)
print(f"Model created with {len(model.locations)} locations")


In [ ]:
# Solve CVRP
print("Solving CVRP...")
max_runtime = 5
result = solve_cvrp(model, max_runtime=max_runtime)
print(f"Solution found:")
print(f"  Total distance: {result.best.distance():.2f}")
print(f"  Number of routes: {result.best.num_routes()}")


In [ ]:
# Extract routes
print("Extracting routes with depot visits...")
routes = extract_routes_with_depots(result)
print(f"Extracted {len(routes)} routes")


In [ ]:
# Calculate load progression
print("Calculating load progression...")
load_progression = calculate_load_progression(routes, node_df)
print(f"Load progression calculated for {len(load_progression)} trips")


In [ ]:
# Route on graph
print("Routing on street network...")
routing_result = route_on_graph(routes, g_ig, node_df, idx_maps)
print(f"Routing complete:")
print(f"  Total segments: {routing_result['routing_stats']['total_segments']}")
print(f"  Successful: {routing_result['routing_stats']['successful']}")
print(f"  Failed: {routing_result['routing_stats']['failed']}")


In [ ]:
# Calculate edge loads
print("Calculating edge loads...")
edge_loads_kg = calculate_edge_loads(routing_result, load_progression, graph, unit='kg')
print(f"Edge loads calculated:")
print(f"  Edges with load (kg): {edge_loads_kg['n_edges_used']}")
print(f"  Max load: {edge_loads_kg['max_load']:.2f} kg")


## Visualization

Visualize edge loads on the street network.


In [ ]:
# Prepare visualization data
def prepare_edge_viz(graph, edge_loads_dict):
    """Prepare edge colors and widths for visualization."""
    edge_load_lookup = edge_loads_dict['edge_loads']
    
    # Get loads for all edges
    loads_with_values = [v for v in edge_load_lookup.values() if v > 0]
    
    if not loads_with_values:
        print("No edges with positive loads found")
        return None, None, None
    
    max_load = max(loads_with_values)
    min_load = min(loads_with_values)
    
    edge_colors = []
    edge_widths = []
    
    for u, v, k in graph.edges(keys=True):
        load = edge_load_lookup.get((u, v, k), 0)
        
        if load > 0:
            # Normalize load to 0-1
            norm_load = (load - min_load) / (max_load - min_load) if max_load > min_load else 0.5
            # Blue (low) -> Yellow (mid) -> Red (high)
            # Use colormap: blue (low) -> yellow -> red (high)
            color = plt.cm.YlOrRd(norm_load)
            # return as hex color
            # color = mpl.colors.rgb2hex(color)
            
            edge_colors.append(color[0:3])
            width = 1 + 4 * norm_load
            edge_widths.append(width)
        else:
            # Unused edges in gray
            edge_colors.append((0.8, 0.8, 0.8))
            edge_widths.append(0.3)
    
    return edge_colors, edge_widths, (min_load, max_load)


In [ ]:
# Visualize edge loads (kg)
fig, ax = plt.subplots(figsize=(15, 15))

edge_colors, edge_widths, load_range = prepare_edge_viz(graph, edge_loads_kg)

if edge_colors:
    ox.plot_graph(
        graph,
        ax=ax,
        node_size=0,
        edge_color=edge_colors,
        edge_linewidth=edge_widths,
        show=False,
        close=False
    )
    
    # Add depot markers
    depot_rows = node_df[node_df["isdepot"]]
    for _, depot in depot_rows.iterrows():
        ax.scatter(depot["x"], depot["y"], c='green', s=200, marker='s', 
                  edgecolors='black', linewidths=2, zorder=5, 
                  label=f'Depot {int(depot["depot_id"])}')
    
    ax.set_title(
        f'Edge Loads (kg)\n'
        f'{edge_loads_kg["n_edges_used"]} edges used, '
        f'max load: {edge_loads_kg["max_load"]:.0f} kg\n'
        f'Load range: {load_range[0]:.0f} - {load_range[1]:.0f} kg',
        fontsize=14
    )
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Cannot visualize: no edge loads found")
